# FreshMart Lab 0: Environment Verification
**Azure Machine Learning (ทางเลือกฉุกเฉินแทน Fabric)**

เป้าหมาย: ยืนยันว่าพร้อมเข้า Lab 1 — **ไม่ต้องวิเคราะห์ธุรกิจในแล็บนี้**

แทร็กนี้ใช้เมื่อ **Fabric Capacity ใช้ไม่ได้** วงจรธุรกิจ FreshMart เหมือนชุด Fabric แต่รันบน Azure ML Compute Instance ด้วย pandas + MLflow

### ก่อนรัน (เช็ก 30 วินาที)
1. Workspace Azure ML เป็นของคลาสหรือของตนเอง — ทำงานในโฟลเดอร์ `Users/<ชื่อคุณ>/freshmart/`
2. Compute instance สถานะ **Running** และเคอร์เนลเป็น **Python 3.10 - SDK v2**
3. มีไฟล์ CSV สามไฟล์จาก `labs/data/` (clone repo หรืออัปโหลดไป `data/raw/`)
4. **อย่ารันหลายเซลล์ซ้อน** จนกว่าเซลล์ก่อนหน้าจะจบ

### ศัพท์ที่ใช้ในแล็บนี้
Workspace ของ Azure ML คือศูนย์กลางทดลองโมเดล จัดเก็บ Data asset, Experiment และ Model  
ชั้น Medallion ในแทร็กนี้คือ**โฟลเดอร์** `data/bronze` / `data/silver` / `data/gold` ไม่ใช่ schema ของ Lakehouse

### ถ้าติด — อ่านก่อนถาม TA
| อาการ | ทำอะไร |
| --- | --- |
| ไม่เจอไฟล์ CSV | อัปโหลดสามไฟล์จาก `labs/data/` ไป `data/raw/` หรือ `git clone` repo นี้บน compute instance |
| Kernel Starting ค้าง | รอ compute เป็น Running แล้วเลือกเคอร์เนลใหม่ |
| `PermissionError` ตอนเขียนไฟล์ | ตรวจว่าทำงานในโฟลเดอร์ของตนเอง ไม่ใช่ Samples ที่อ่านอย่างเดียว |
| AssertionError จำนวนแถว | ไม่ผ่าน Lab 0 — อย่าข้ามไป Lab 1 |


### เซลล์ถัดไป: ฟังก์ชันโหลดข้อมูล

โค้ดด้านล่างนิยาม `load_table_or_csv` และ `save_layer` ให้แล้ว — **รันครั้งเดียวแล้วใช้ต่อทุก Lab**

- อ่าน parquet ใน `data/bronze` ก่อน ถ้ายังไม่มีค่อยอ่าน CSV จาก `data/raw/` หรือ `labs/data/`
- เขียนชั้น Medallion เป็นไฟล์ในโฟลเดอร์ `data/` ของ workspace
- ไม่ต้องแก้โค้ดนี้


In [ ]:
from pathlib import Path
import pandas as pd

BRONZE_TRANSACTIONS = "bronze/transactions"
BRONZE_CUSTOMERS = "bronze/customers"
SILVER_CUSTOMER_FEATURES = "silver/customer_features"
GOLD_PREDICTIONS = "gold/freshmart_predictions"
ASSET_BRONZE_TRANSACTIONS = "bronze-transactions"
ASSET_BRONZE_CUSTOMERS = "bronze-customers"
ASSET_SCORING_BATCH = "scoring-batch"
ASSET_SILVER_FEATURES = "silver-customer-features"
ASSET_GOLD_PREDICTIONS = "gold-freshmart-predictions"
TABLE_TO_ASSET = {
    BRONZE_TRANSACTIONS: ASSET_BRONZE_TRANSACTIONS,
    BRONZE_CUSTOMERS: ASSET_BRONZE_CUSTOMERS,
    SILVER_CUSTOMER_FEATURES: ASSET_SILVER_FEATURES,
    GOLD_PREDICTIONS: ASSET_GOLD_PREDICTIONS,
}
CSV_TO_ASSET = {
    "freshmart_transactions.csv": ASSET_BRONZE_TRANSACTIONS,
    "freshmart_customers.csv": ASSET_BRONZE_CUSTOMERS,
    "freshmart_scoring_batch.csv": ASSET_SCORING_BATCH,
}
RAW_FILES = (
    "freshmart_transactions.csv",
    "freshmart_customers.csv",
    "freshmart_scoring_batch.csv",
)


def _first_existing(paths):
    for path in paths:
        candidate = Path(path)
        if candidate.exists() and candidate.is_file():
            return candidate
    return None


def _writable_dir(path: Path) -> Path | None:
    try:
        path.mkdir(parents=True, exist_ok=True)
        probe = path / ".write_test"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        return path.resolve()
    except OSError:
        return None


def resolve_raw_csv(file_name: str) -> Path:
    found = _first_existing([
        Path("data") / "raw" / file_name,
        Path("data") / file_name,
        Path("../data") / "raw" / file_name,
        Path("../data") / file_name,
        Path("../../labs/data") / file_name,
        Path("../labs/data") / file_name,
        Path("labs/data") / file_name,
        Path(file_name),
    ])
    if found is None:
        raise FileNotFoundError(
            f"Cannot find {file_name}. Upload it to data/raw/ next to the notebook "
            "or clone this repo so labs/data/ is available."
        )
    return found


def load_csv(file_name: str) -> pd.DataFrame:
    asset_name = CSV_TO_ASSET.get(file_name)
    if asset_name:
        frame = load_data_asset(asset_name)
        if frame is not None:
            return frame
    found = resolve_raw_csv(file_name)
    print(f"Loaded CSV: {found}")
    return pd.read_csv(found)


def load_data_asset(asset_name: str):
    """Load a registered Azure ML data asset, or return None."""
    try:
        from azure.ai.ml import MLClient
        from azure.identity import DefaultAzureCredential

        ml_client = MLClient.from_config(credential=DefaultAzureCredential())
        asset = ml_client.data.get(name=asset_name, label="latest")
        path = asset.path
        print(f"Loaded data asset {asset_name} v{asset.version}: {path}")
        if str(path).lower().endswith(".parquet"):
            return pd.read_parquet(path)
        return pd.read_csv(path)
    except Exception as exc:
        print(f"Data asset '{asset_name}' unavailable ({exc})")
        return None


def register_data_asset(name: str, path, description: str = "") -> bool:
    """Register a file as an Azure ML data asset. Returns True when saved."""
    try:
        from azure.ai.ml import MLClient
        from azure.ai.ml.entities import Data
        from azure.ai.ml.constants import AssetTypes
        from azure.identity import DefaultAzureCredential

        ml_client = MLClient.from_config(credential=DefaultAzureCredential())
        asset = Data(
            name=name,
            path=str(path),
            type=AssetTypes.URI_FILE,
            description=description or f"FreshMart {name}",
        )
        created = ml_client.data.create_or_update(asset)
        print(f"Registered data asset {created.name} v{created.version}")
        return True
    except Exception as exc:
        print(f"Could not register data asset '{name}' ({exc})")
        return False


def resolve_artifact_root() -> Path:
    for candidate in (
        Path("data"),
        Path("../data"),
        Path("labs-azureml/data"),
    ):
        ready = _writable_dir(candidate)
        if ready is not None:
            return ready
    fallback = Path("data")
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback.resolve()


ARTIFACT_ROOT = resolve_artifact_root()


def layer_path(layer: str, stem: str, suffix: str = ".parquet") -> Path:
    folder = ARTIFACT_ROOT / layer
    folder.mkdir(parents=True, exist_ok=True)
    return folder / f"{stem}{suffix}"


def load_table_or_csv(table_name: str, file_name: str) -> pd.DataFrame:
    asset_name = TABLE_TO_ASSET.get(table_name)
    if asset_name:
        frame = load_data_asset(asset_name)
        if frame is not None:
            return frame
    layer, _, stem = table_name.partition("/")
    parquet = layer_path(layer, stem)
    if parquet.exists():
        frame = pd.read_parquet(parquet)
        print(f"Loaded parquet {parquet}: {len(frame):,} rows")
        return frame
    csv_fallback = ARTIFACT_ROOT / layer / f"{stem}.csv"
    if csv_fallback.exists():
        frame = pd.read_csv(csv_fallback)
        print(f"Loaded CSV artifact {csv_fallback}: {len(frame):,} rows")
        return frame
    print(f"Layer file '{table_name}' not found. Falling back to published CSV.")
    return load_csv(file_name)


def save_layer(frame: pd.DataFrame, table_name: str) -> Path:
    layer, _, stem = table_name.partition("/")
    parquet = layer_path(layer, stem)
    try:
        frame.to_parquet(parquet, index=False)
        print(f"Wrote {parquet} ({len(frame):,} rows)")
        return parquet
    except Exception as exc:
        csv_path = layer_path(layer, stem, suffix=".csv")
        frame.to_csv(csv_path, index=False)
        print(f"Parquet unavailable ({exc}). Wrote {csv_path}")
        return csv_path


### ตรวจแพ็กเกจที่ต้องใช้

**โค้ดนี้ทำอะไร:** ตรวจว่า pandas / scikit-learn / matplotlib พร้อมบน compute instance

ถ้าขาด เซลล์จะติดตั้งให้แบบเงียบ — รอบแรกอาจใช้เวลา 1–2 นาที


In [ ]:
import importlib.util

required = ["pandas", "sklearn", "matplotlib", "seaborn", "mlflow", "pyarrow", "azure.ai.ml"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "scikit-learn", "matplotlib", "seaborn", "mlflow", "pyarrow", "azure-ai-ml", "azureml-fsspec"])
    print("Installed:", ", ".join(missing))
else:
    print("Python packages ready:", ", ".join(required))


### สร้างชั้น Bronze จากไฟล์ข้อมูลต้นทาง (CSV)

**โค้ดนี้ทำอะไร:** คัดลอกธุรกรรมและสมาชิกจาก CSV ไปเป็นไฟล์ parquet ใน `data/bronze/`

รันครั้งเดียวหลังมีไฟล์ CSV — รันซ้ำได้ (เขียนทับชุดเดิม)

**สิ่งที่ควรเห็น**
- `bronze/transactions` ประมาณ **3,000** แถว
- `bronze/customers` ประมาณ **1,500** แถว
- ข้อความ `Bronze layers ready`


In [ ]:
required = [
    "freshmart_transactions.csv",
    "freshmart_customers.csv",
    "freshmart_scoring_batch.csv",
]
missing = []
for name in required:
    asset_name = CSV_TO_ASSET.get(name)
    if asset_name and load_data_asset(asset_name) is not None:
        continue
    try:
        resolve_raw_csv(name)
    except FileNotFoundError:
        missing.append(name)
if missing:
    raise FileNotFoundError(
        "ไม่พบไฟล์หรือ Data asset: "
        + ", ".join(missing)
        + " — สร้าง Data asset จากหน้า Data หรืออัปโหลดจาก labs/data/"
    )

df_tx = load_csv("freshmart_transactions.csv")
df_cust = load_csv("freshmart_customers.csv")
tx_path = save_layer(df_tx, BRONZE_TRANSACTIONS)
cust_path = save_layer(df_cust, BRONZE_CUSTOMERS)
print("bronze/transactions:", len(df_tx))
print("bronze/customers:", len(df_cust))
print("Artifact root:", ARTIFACT_ROOT)
print("Bronze layers ready")


### โหลด Bronze แล้วดูตัวอย่างแถว

**โค้ดนี้ทำอะไร:** อ่านชั้น Bronze แล้วพิมพ์จำนวนแถวและ 5 แถวแรก

**สิ่งที่ควรเห็น**
- Transactions ประมาณ **3,000** แถว
- Customers ประมาณ **1,500** แถว


In [ ]:
df_tx = load_table_or_csv(BRONZE_TRANSACTIONS, "freshmart_transactions.csv")
df_cust = load_table_or_csv(BRONZE_CUSTOMERS, "freshmart_customers.csv")

print(f"Transactions rows: {len(df_tx):,}")
print(f"Customers rows: {len(df_cust):,}")
print("Transaction columns:", list(df_tx.columns))
print("Customer columns:", list(df_cust.columns))
display(df_tx.head(5))
display(df_cust.head(5))


### ลงทะเบียน Data asset ใน Azure ML

**โค้ดนี้ทำอะไร:** ลงทะเบียนชั้น Bronze และชุดทำนายเป็น Data asset ใน workspace  
ถ้าคุณสร้างจากหน้า **Data** ใน studio แล้ว เซลล์นี้จะเพิ่มเวอร์ชันใหม่

**สิ่งที่ควรเห็น:** เมนู **Data** มี `bronze-transactions`, `bronze-customers`, `scoring-batch`


In [ ]:
register_data_asset(ASSET_BRONZE_TRANSACTIONS, tx_path, "FreshMart bronze transactions")
register_data_asset(ASSET_BRONZE_CUSTOMERS, cust_path, "FreshMart bronze customers")
try:
    scoring_path = resolve_raw_csv("freshmart_scoring_batch.csv")
    register_data_asset(ASSET_SCORING_BATCH, scoring_path, "FreshMart unlabeled scoring batch")
except FileNotFoundError:
    print("Create scoring-batch from the Data page if the CSV is not on this compute.")


### จุดตรวจอัตโนมัติ

**โค้ดนี้ทำอะไร:** ถ้าจำนวนแถวไม่ตรง จะ `raise AssertionError` และหยุด

- ผ่านแล้วจะพิมพ์ `Lab 0 verification passed`
- ไม่ผ่าน แปลว่าสภาพแวดล้อมยังไม่พร้อม — **ถาม TA พร้อมคัดลอกข้อความ error ทั้งบรรทัด**


In [ ]:
if len(df_tx) != 3000:
    raise AssertionError(
        f"คาดว่าธุรกรรม 3,000 แถว แต่ได้ {len(df_tx):,} — ตรวจ data/raw หรือ labs/data"
    )
if len(df_cust) != 1500:
    raise AssertionError(
        f"คาดว่าสมาชิก 1,500 แถว แต่ได้ {len(df_cust):,} — ตรวจ data/raw หรือ labs/data"
    )
print("Lab 0 verification passed")
